## v0.1


In [2]:
MODEL = "gpt-4o-mini-2024-07-18"
Q_EXTRACTION_PROMPT = "q_extraction_v0.1"
Q_QUALITY_PROMPT = "q_judge_v0.1"
IDS_LIST_NAME = "initial_ids_v0.1"
CANDIDATES_NAME = "spotify_candidates_v0.1"

MIN_Q_MATCH_RATE = 0.9

OUTFILE = f"../output/a_extraction/inputs/{MODEL}.{Q_EXTRACTION_PROMPT}.{Q_QUALITY_PROMPT}/{IDS_LIST_NAME}.tsv"

### Run

In [4]:
import pandas as pd

quality_eval_file = f"../output/q_quality/parsed/{MODEL}/{Q_QUALITY_PROMPT}/{IDS_LIST_NAME}.tsv"
q_extraction_file = f"../output/q_extraction/parsed/{MODEL}/{Q_EXTRACTION_PROMPT}/{IDS_LIST_NAME}.tsv"
input_file = f"../output/data/{CANDIDATES_NAME}.tsv"

df_quality = pd.read_csv(quality_eval_file, sep="\t")
df_input = pd.read_csv(input_file, sep="\t")
df_extraction = pd.read_csv(q_extraction_file, sep="\t")

df = (
    df_quality
        .merge(df_extraction, on="podcast_id", how="left")
        .merge(df_input, on="podcast_id", how="left")
)

In [5]:
df[["info_seeking", "self_contained"]].value_counts(dropna=False)

info_seeking  self_contained
no            no                15
yes           yes               13
              no                 6
Name: count, dtype: int64

In [6]:
# Print some good questions:
pd.set_option("display.max_colwidth", None)
(
    df
        .query("info_seeking == 'yes' & self_contained == 'yes'")
        .sort_values("info_seeking_logprob", ascending=False)
        .head(20)
        [["question", "response"]] # "info_seeking_logprob", 
)

,question,response
1,"""What is a UIP pattern?""","Self-contained: The question is clear and understandable on its own, as it defines a specific term (UIP pattern) without needing additional context. Verdict: yes. \nInformation-seeking: The question asks for a general definition or explanation of a concept, which is factual knowledge rather than personal experience. Verdict: yes."
6,What are the advantages of staying or returning to Real Madrid?,"Self-contained: The question is clear and understandable on its own, as it specifies the context of staying or returning to Real Madrid without needing prior conversation. Verdict: yes. \nInformation-seeking: The question seeks general knowledge about the advantages of a decision related to a football club, rather than personal experiences of the speakers. Verdict: yes."
8,"""Who were these armed insurgents who were taking them hostage?""","Self-contained: The question is clear and understandable on its own, as it specifies ""armed insurgents"" and the context of taking hostages without needing prior conversation. Verdict: yes. \nInformation-seeking: The question seeks factual knowledge about the identity of the armed insurgents, rather than personal experiences of the speakers. Verdict: yes."
9,"""If anyone knows any, any good like editing tips or editing software like for the for the cheap because I'm kind of like I'm kind of broke right.""","Self-contained: The question is somewhat ambiguous as it lacks clarity and specificity about the type of editing tips or software being sought. However, it does convey a general request for information. Verdict: yes. \nInformation-seeking: The question is seeking general factual knowledge about editing tips and affordable software, rather than personal experiences. Verdict: yes."
11,"""What games are you looking forward to the most from Platinum Games?""","Self-contained: The question specifies ""Platinum Games"" and asks about upcoming games, which provides enough context for someone unfamiliar with the conversation to understand that it pertains to anticipated releases from a specific game developer. Verdict: yes. \nInformation-seeking: The question seeks general information about upcoming games from a particular developer, rather than personal experiences of the speakers. Verdict: yes."
24,What is called AMSMR style?,"Self-contained: The question is clear and does not require prior context to understand what is being asked. It specifies a term (AMSMR style) and seeks information about it. Verdict: yes. \nInformation-seeking: The question is looking for a definition or explanation of a specific term, which aligns with seeking general factual knowledge. Verdict: yes."
13,"""Do you have any comments or advice regarding future pre-med applicants who either come from business or marketing or just kind of is transitioning right now?""","Self-contained: The question is somewhat ambiguous because it references ""future pre-med applicants"" and their backgrounds without providing context on what specific comments or advice might be relevant. However, it does imply a general inquiry about transitioning into pre-med, which could be understood without prior conversation. Verdict: yes.\n\nInformation-seeking: The question seeks general advice and comments about the transition of applicants from business or marketing to pre-med, which is a broader topic and not focused on personal experiences of the speakers. Verdict: yes."
17,How can we create communities and ritual to reinvigorate us?,"Self-contained: The question is clear and can be understood without prior context, as it asks about the general concept of creating communities and rituals. Verdict: yes. \nInformation-seeking: The question seeks general knowledge on the topic of community building and rituals, rather than personal experiences. Verdict: yes."
23,"""Can guys and girls be friends when one of them catches feelings?""","Self-contained: The question is clear and understandable on its own, as i

In [7]:
# Keep the good ones:
df_keep = df.query(
    f"match_rate_q > {MIN_Q_MATCH_RATE} & info_seeking == 'yes' & self_contained == 'yes'").copy()

# Build the input as concatenation of podcast and question with tags:
df_keep["a_extraction_input"] = "<podcast>\n\t" + df_keep["text"].str.strip() + \
        "\n</podcast>\n\n<question>\n\t" + \
        df_keep["question"].str.strip() + "\n</question>"

In [9]:
# print first 2 to check it's good:
for i, row in df_keep.head(2).iterrows():
    print(f'{row["podcast_id"]=}')
    print(row["a_extraction_input"])
    print("\n\n")

row["podcast_id"]='podcasts-audio/3/F/show_3FA7poT1MNGUp50VjkzbkB/72io6qBtwMjzU4B3xMGHFh'
<podcast>
	Hello and welcome back to Songs for FRCR. We're back after a short hiatus. We're back after a short hiatus. There was no episode last week, mainly because we had some issues publishing the podcast on certain platforms. But that's all sorted and we are back now with a brand new episode on the most commonly requested topic on Songs for FRCR. We've had so many emails and tweets about this. It's everyone's favourite, the idiopathic interstitial pneumonias. Pretty much everyone's reaction when that topic is mentioned, but it doesn't need to be that way. It's not a difficult topic and we'll try and make it a lot easier for you in this episode. We're trying something new this week. As well as the podcast episode, we've created a poster or a handout, call it what you will, to do with idiopathic interstitial pneumonias. The idea is you print it off, stick it on your bedroom wall and every mornin

In [10]:
# Save to file:
from pathlib import Path

Path(OUTFILE).parent.mkdir(parents=True, exist_ok=True)
df_keep[["podcast_id", "a_extraction_input"]].to_csv(OUTFILE, sep="\t", index=False)

In [11]:
print(f"{len(df_keep)}/{len(df)} questions kept")
print(f"Saved to: {OUTFILE}")

13/34 questions kept
Saved to: ../output/a_extraction/inputs/gpt-4o-mini-2024-07-18.q_extraction_v0.1.q_judge_v0.1/initial_ids_v0.1.tsv


In [12]:
# Check it was saved correctly:
df_read = pd.read_csv(OUTFILE, sep="\t")
df_read.shape #, df_read.head(2)

(13, 2)

--------------------